In [ ]:
# Install the GitHub CardPy checkout into this notebook's Python environment.
from pathlib import Path
import subprocess
import sys

CARDPY_REPO = Path.home() / "Documents" / "GitHub" / "CarDpy"
if not (CARDPY_REPO / "pyproject.toml").is_file():
    raise FileNotFoundError(f"CardPy checkout not found: {CARDPY_REPO}")

# Equivalent to: cd ~/Documents/GitHub/CarDpy && python -m pip install -e .
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    cwd=str(CARDPY_REPO),
)


In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [ ]:
from pathlib import Path
import os
import sys

# Use the current GitHub checkout rather than Scripts/CarDpy-master.
CARDPY_REPO = Path.home() / 'Documents' / 'GitHub' / 'CarDpy'
if not (CARDPY_REPO / 'cardpy' / '__init__.py').is_file():
    raise FileNotFoundError(f'CardPy checkout not found: {CARDPY_REPO}. Update CARDPY_REPO above.')

# A kernel that already imported legacy CardPy must be restarted first.
for module_name, module in tuple(sys.modules.items()):
    if module_name == 'cardpy' or module_name.startswith('cardpy.'):
        module_file = getattr(module, '__file__', None)
        if module_file and not Path(module_file).resolve().is_relative_to(CARDPY_REPO.resolve()):
            raise RuntimeError('A different CardPy copy is already loaded. Restart the kernel and run this notebook again.')
sys.path.insert(0, str(CARDPY_REPO))
import cardpy
print(f'Using CardPy: {Path(cardpy.__file__).resolve()}')
import glob
from smart_health_config import get_config
# Compatibility process preserves the original sequence and output numbering.
from smart_health_processing import process
from smart_health_inputs import resolve_blip_directory


In [ ]:
# Shared original settings live in smart_health_config.py.
config = get_config("blip")


In [ ]:
volunteer = 'Aortic Stenosis Patient 007' # 'SCH_vol_XX or SCH_Patient_XX' 
distortion = 'BlipDown' # or 'BlipUp'
sl_id = 'Even' # Even or Odd
even_odd_mode = False # True or False if acquired even/odd slices seperately on will handle the even/odd 

main_data_folder = '/Volumes/T7/Smart_Health_Reorganized/00_Original_Data' # path to the main data folder where the volunteer folder is in
main_output_folder = '/Volumes/T7/Smart_Health_Reorganized/01_Processed_Data' # path to the main output folder where the volunteer folder is in

In [ ]:
study_root = os.path.join(main_data_folder, volunteer)
# Smart Health convention: Blip Up = FH/LR/AP; Blip Down = HF/RL/PA. Checks one DICOM per series.
input_directory = resolve_blip_directory(
    study_root, distortion, even_odd_mode=even_odd_mode, sl_id=sl_id,
)
config["dicom_subpath"] = os.path.relpath(input_directory, study_root)
print(f"Selected {distortion} DICOM series: {input_directory}")

out_path = os.path.join(main_output_folder, volunteer, 'Blip_Down')
if even_odd_mode:
    out_path += '_' + sl_id
config["output"]["directory"] = out_path


In [ ]:
results = process(study_root, config)
